# ***Part 1: Data Processing for Vapor Pressure Prediction***

## **Section 0.0 Installing ChemProp and Dependencies**

Information on this is available on the chemprop website: https://chemprop.readthedocs.io/en/v1.7.1/installation.html

In [ ]:
## If using conda, you can create an environment with the necessary packages by going to your terminal and running these lines.

#conda create -n chemprop python=3.8
#conda install -c conda-forge rdkit
#pip install git+https://github.com/bp-kelley/descriptastorus
#pip install chemprop

## **Section 1.0 Reading Data from CSV for Pandas Data Frame**

The data was obtained from JOEHUSSEINMAMA (HAHAHA... ghost account?) on Kaggle and they site that the compounds were obtained from *Yaw's Handbook of Vapor Pressure*. The link to the Kaggle page is given below. I spot checked the data with a pdf of *Yaw's Handbook of Vapor Pressure* and it held up for the number of points I selected.

(https://www.kaggle.com/datasets/joehusseinmama/antoine-coefficients-for-vapor-pressure?resource=download&select=complete_organic_table.csv)

The form of the Antoine equation referenced in Yaw's Handbook is shown below.

$$\log_{10}P = A - \frac{B}{T + C}$$
$$P\quad[=] \quad\text[{mmHg}]\quad[\approx]\quad\text[{torr}]$$
$$T \quad[=] \quad[^\circ\text{C}]$$

In [ ]:
import pandas as pd

# Load the dataset, you may need to adjust the path based on where your file is located
data_path = r'../Datasets/Raw_Dataset/complete_organic_table.csv'
df = pd.read_csv(data_path)
display(df.tail(10))

In [ ]:
import pubchempy as pcp
import pandas as pd
import requests
from tqdm import tqdm

tqdm.pandas()  # Enable tqdm for pandas

def cas_to_smiles(cas_number):
    """Retrieve SMILES from PubChem using CAS number."""
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{cas_number}/property/IsomericSMILES/JSON"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            smiles = response.json()['PropertyTable']['Properties'][0]['SMILES']
            return smiles
    except:
        pass
    return None

def name_to_smiles(name):
    """Retrieve SMILES from PubChem using compound name."""
    try:
        compound = pcp.get_compounds(name, 'name')
        if compound:
            return compound[0].canonical_smiles
    except:
        pass
    return None

def subset_with_smiles(df, name_col="NAME", cas_col="CAS"):
    """
    Adds a 'SMILES' column to the dataframe by first querying PubChem with the CAS number,
    then falling back to the compound name if CAS fails. Returns only rows where SMILES is found.
    Shows a tqdm progress bar.
    """
    df = df.copy()
    
    def get_smiles(row):
        # Try CAS first if available
        cas = row[cas_col]
        if cas not in [None, "---", ""]:
            smiles = cas_to_smiles(cas)
            if smiles:
                return smiles
        # Fall back to name
        name = row[name_col]
        if name not in [None, "---", ""]:
            return name_to_smiles(name)
        return None

    # Apply with progress bar
    df["SMILES"] = df.progress_apply(get_smiles, axis=1)
    
    # Return only rows where SMILES was successfully retrieved
    return df[df["SMILES"].notna()]


In [ ]:
subset_df = subset_with_smiles(df, name_col="NAME", cas_col="CAS")

# Number of successful SMILES retrievals
num_success = len(subset_df)

# Number of unsuccessful attempts
num_fail = len(df) - num_success

print(f"Number of successful SMILES retrievals: {num_success}")
print(f"Number of unsuccessful attempts: {num_fail}")

# Show first few successful rows
print(subset_df.head())

In [ ]:
subset_df.to_csv("../Datasets/Curated Dataset(s)/CAS_and_IUPAC_pull.csv")

In [ ]:
vp_df = calculate_vapor_pressures(subset_df)

In [ ]:
# vp_df is the vapor pressure dataset (with multiple rows per compound)
# df is your master reference dataframe with molecule metadata

percent, low_vp_subset = low_vp_molecules(vp_df, df, threshold=1.0)

print(f"{percent:.2f}% of molecules have at least one VP < 1 mmHg")
print(low_vp_subset.head())

## **Section 2.0 Creating two subsets of the data based on having CAS numbers or not (Most improvement to total data can be made here)**

This will be important when adding corresponding SMILES strings to the data. We will need to reference some sort of chemical identifier like CAS number or IUPAC name (preferred due to stereoisomer considerations) in order to find the associated SMILES string.

In [ ]:
cas_subset = df[df['CAS'].notnull() & (df['CAS'] != '---')]
nocas_subset = df[df['CAS'].isnull() | (df['CAS'] == '---')]

print(f"Original DataFrame size: {len(df)}")
print(f"Subset DataFrame size: {len(cas_subset)}")
print(f"Data without SMILES strings: {len(nocas_subset)}")
print(f"Percentage of data retained: {(len(cas_subset)/len(df))*100:.1f}%")

## **Section 3.0 Adding a SMILES column by requesting it from Pubchem using the associated CAS number**

Just for the sake of having all of the SMILES strings on hand to begin with, I'm obtaining the SMILES strings first. I will later downselect this dataset later based on criteria I will described later.

In [ ]:
# # Dependencies necessary for obtaining a SMILES string from a CAS number

%pip install requests
%pip install tqdm

### S3.1 Function to get SMILES from CAS number (not terribly robust, but gets the job done)

In [ ]:
import requests
from rdkit import Chem

def cas_to_smiles(cas_number):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{cas_number}/property/IsomericSMILES/JSON"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            smiles = response.json()['PropertyTable']['Properties'][0]['SMILES']
            return smiles
        else:
            return None
    except:
        return None

### **FUNCTIONS TO DELETE IF NOT USED BY END**

#### **(DELETE IF NOT USED LATER)** Function to get SMILES strings from IUPAC names (pubchem)

In [ ]:
import requests

def iupac_to_smiles(compound_name):
    """
    Converts IUPAC name to SMILES string using PubChem's API.
    
    Parameters:
        compound_name (str): IUPAC name of the compound
        
    Returns:
        str or None: SMILES string if found, else None
    """
    try:
        # Step 1: Get CID from compound name
        cid_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{compound_name}/cids/JSON"
        cid_response = requests.get(cid_url, timeout=10)
        cid_response.raise_for_status()

        cid_data = cid_response.json()
        cid = cid_data.get('IdentifierList', {}).get('CID', [None])[0]
        if not cid:
            print(f"No CID found for compound: {compound_name}")
            return None

        # Step 2: Get SMILES from CID
        smiles_url = (
            f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/"
            f"CanonicalSMILES,IsomericSMILES/JSON"
        )
        smiles_response = requests.get(smiles_url, timeout=10)
        smiles_response.raise_for_status()

        smiles_data = smiles_response.json()
        props = smiles_data.get('PropertyTable', {}).get('Properties', [{}])[0]
        smiles = props.get('SMILES') or props.get('CanonicalSMILES')

        return smiles

    except requests.RequestException as e:
        print(f"Request error for {compound_name}: {e}")
        return None

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas()

low_vp_subset['SMILES'] = low_vp_subset['CAS'].progress_apply(iupac_to_smiles)

print(f"Successfully retrieved SMILES for {low_vp_subset['SMILES'].notna().sum()} compounds")
print(f"Failed to retrieve SMILES for {low_vp_subset['SMILES'].isna().sum()} compounds")

#### **(DELETE IF NOT USED LATER)** Function to fetch SMILES from CAS number then tries IUPAC to SMILES if that failed.

In [ ]:
def fetch_smiles(row):
    """
    Attempts to fetch SMILES first using CAS number, falls back to compound name if CAS fails
    
    Parameters:
        row: pandas Series containing 'CAS' and 'NAME' columns
        
    Returns:
        str or None: SMILES string if found, else None
    """
    # First try CAS if it exists and is valid
    if pd.notna(row['CAS']) and row['CAS'] != '---':
        try:
            # Try CAS lookup first
            cid_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{row['CAS']}/cids/JSON"
            cid_response = requests.get(cid_url, timeout=10)
            if cid_response.status_code == 200:
                cid_data = cid_response.json()
                cid = cid_data.get('IdentifierList', {}).get('CID', [None])[0]
                if cid:
                    smiles_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/CanonicalSMILES,IsomericSMILES/JSON"
                    smiles_response = requests.get(smiles_url, timeout=10)
                    if smiles_response.status_code == 200:
                        smiles_data = smiles_response.json()
                        props = smiles_data.get('PropertyTable', {}).get('Properties', [{}])[0]
                        smiles = props.get('IsomericSMILES') or props.get('CanonicalSMILES')
                        if smiles:
                            return smiles
        except requests.RequestException:
            pass  # Silently fail and try NAME instead
    
    # If CAS lookup failed or wasn't available, try NAME
    try:
        return iupac_to_smiles(row['NAME'])
    except requests.RequestException as e:
        print(f"Request error for {row['NAME']}: {e}")
        return None

### **S3.2** Applying the **cas_to_smiles** function from above to get the SMILES strings for the **cas_subset** data frame

This block of code took around 30 minutes on my laptop connected via ethernet cable to the internet at the University of Missouri - Columbia (i.e. good connection). By running this code, you are requesting information from the Pubchem website. The slower your connection to the internet is (higher your latency), the slower this requesting process will be. Keep those things in mind before running this cell.

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas()

cas_subset['SMILES'] = cas_subset['CAS'].progress_apply(cas_to_smiles)

print(f"Successfully retrieved SMILES for {cas_subset['SMILES'].notna().sum()} compounds")
print(f"Failed to retrieve SMILES for {cas_subset['SMILES'].isna().sum()} compounds")

#### Performing a data balance as a sanity check

Previously we had a **cas_subset** and a **nocas_subset** of the original dataframe created from *Yaw's Handbook of Vapor Pressure*, but we just ran the **cas_subset** through our **cas_to_smiles** function to get all the SMILES string we could from the Pubchem website. Not all CAS number entrires in the **cas_subset** could be translated to a SMILES string. Of the **9919 total compounds with a CAS number**, only **8194 successfully pulled SMILES strings**. The remaining 1725 compounds are combined with the **nocas_subset** below to ensure all of our data is preserved if we want to go back and reprocess the bad eggs later. Due to the somewhat convoluted transfer of data that was occuring, it's good to do a balance on original data and the two subset we just created to ensure we didn't make a mistake.

In [ ]:
import os
output_dir = '../Datasets/Curated Dataset(s)'

# Store rows that failed SMILES retrieval
failed_smiles_mask = cas_subset['SMILES'].isna()
rows_to_move = cas_subset[failed_smiles_mask].copy()

# Remove rows with no SMILES from cas_subset
cas_subset = cas_subset[~failed_smiles_mask]

# Add failed rows to nocas_subset
nocas_subset = pd.concat([nocas_subset, rows_to_move])
noSMILES_subset = nocas_subset #Naming is more appropriate because we just added entries with CAS numbers but no SMILES to this
SMILES_subset = cas_subset
SMILES_subset.to_csv(os.path.join(output_dir,'SMILES_subset.csv'), index=False) #Save the subset to a csv for ease of loading.
nocas_subset.to_csv(os.path.join(output_dir, 'nocas_subset_with_failed_smiles.csv'), index=False) #Save the nocas subset to a csv for ease of loading.

# Print summary
print(f"Cas subset size after removing failed SMILES: {len(cas_subset)}")
print(f"No-cas subset size after adding failed SMILES: {len(nocas_subset)}")
print(f"Total rows (should match original): {len(cas_subset) + len(nocas_subset)}")

At this point, there are 2 separate data sets.

**cas_subset**: is the data frame which contains entries which had CAS numbers to begin with and their corresponding SMILES strings were successfully requested from Pubchem.

**noSMILES_subset**: is the data frame which contains entries where one of two things was true.
- CAS number was **NOT** available (this also includes the last rows in the data frame which contain information about the temperature and pressure units)
- A CAS number **WAS** available, **BUT** a SMILES string could not be successfully obtained in the rudimentary way from Pubchem


## **Section 4.0 Preliminary filtering of the dataset based on SMILES strings and Antoine coefficient confidence (described below)**

### ***Section 4.1** Filtering based on a *code* corresponding to how the Antoine coefficients were determined*

In Yaw's Handbook, it is noted that the codes correspond with the descriptions below.

**Code = 1** -> "A code of 1 in the tabulation is based on experimental data."

**Code = 2 ->** "A code of 2 indicates an estimate... [in relation to code 3 estimates] The estimates with a code of 2 are more accurate."

**Code = 3** -> "The estimates with a code of 3 should be considered rough approximations."

I'm only utlizing code 1 and 2 (and their overlap), for now.

In [ ]:
import pandas as pd
output_dir = '../Datasets/Curated Dataset(s)'
SMILES_subset = pd.read_csv('../Datasets/Curated Dataset(s)/CAS_and_IUPAC_pull.csv') # Load the cas subset with smiles

SMILES_subset['code'] = SMILES_subset['code'].str.strip() # Making sure there is no white space in the entries
SMILES_subset['code'] = SMILES_subset['code'].astype(str) # Convert this column to strings, it was acting weird.


SMILES_subset_c12 = SMILES_subset[SMILES_subset['code'].str.contains('1') | SMILES_subset['code'].str.contains('2')]
working_small_c3 = SMILES_subset[SMILES_subset['code'].str.contains('3')]

print('Datapoints categorized as code 1 or 2 or 1,2:', len(SMILES_subset_c12))
print('Datapoints categorized as code 3:', len(working_small_c3))

### ***Section 4.2** Removing ions, inorganics, organometallics, invalid SMILES, and ensuring all SMILES string are canonical. Additionally, any molecule that contains an element where the element occurrs in less than 1% of the molecules in the dataset is removed.*

In [ ]:
import pandas as pd
from rdkit import Chem
from collections import Counter
import os

def clean_smiles_dataset(df, output_path, threshold=0.01):
    """
    Cleans a dataset containing SMILES strings:
      - Canonicalizes SMILES
      - Removes invalid molecules
      - Removes inorganic molecules (no carbon)
      - Removes organometallic molecules
      - Removes ions
      - Drops duplicates
      - Removes molecules with elements appearing in < threshold fraction of dataset
      - Saves cleaned DataFrame as CSV with '_clean' suffix

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe containing a SMILES column ("SMILES" or "smiles").
    output_path : str
        Path to save the cleaned CSV (suffix '_clean.csv' will be added automatically).
    threshold : float
        Minimum fraction of molecules required for an element to be kept (default=0.01 = 1%).

    Returns
    -------
    cleaned_df : pd.DataFrame
        The cleaned DataFrame.
    atom_summary : pd.DataFrame
        Summary of atom counts across dataset.
    """
    # Detect column
    smiles_col = None
    for candidate in ["SMILES", "smiles"]:
        if candidate in df.columns:
            smiles_col = candidate
            break
    if smiles_col is None:
        raise ValueError("No 'SMILES' or 'smiles' column found in DataFrame.")
    
    # --- Helpers ---
    def canonicalize_smiles(smiles):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return None
            return Chem.MolToSmiles(mol, canonical=True)
        except:
            return None

    def is_organic(mol):
        return any(atom.GetSymbol() == "C" for atom in mol.GetAtoms())

    def is_organometallic(mol):
        metal_atomic_numbers = set([
            3,4,11,12,13,19,20,21,22,23,24,25,26,27,28,29,30,
            31,37,38,39,40,41,42,43,44,45,46,47,48,49,50,
            55,56,57,72,73,74,75,76,77,78,79,80,81,82,83,
            87,88,89
        ])
        return any(atom.GetAtomicNum() in metal_atomic_numbers for atom in mol.GetAtoms())

    def is_ion(mol):
        return Chem.GetFormalCharge(mol) != 0

    def analyze_elements(df, smiles_column):
        atom_type_counts = Counter()
        molecule_atom_presence = Counter()
        for smi in df[smiles_column]:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                continue
            mol = Chem.AddHs(mol)
            atom_symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
            atom_type_counts.update(atom_symbols)
            molecule_atom_presence.update(set(atom_symbols))
        return pd.DataFrame({
            "Total_Atom_Count": pd.Series(atom_type_counts),
            "Molecule_Count": pd.Series(molecule_atom_presence)
        }).fillna(0).astype(int)

    def remove_rare_elements(df, smiles_col, threshold):
        atom_summary = analyze_elements(df, smiles_column=smiles_col)
        total_molecules = len(df)
        rare_elements = atom_summary[
            atom_summary["Molecule_Count"] / total_molecules < threshold
        ].index.tolist()
        print(f"Rare elements (<{threshold*100:.1f}% of molecules): {rare_elements}")
        mask = []
        for smi in df[smiles_col]:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                mask.append(False)
                continue
            atoms = {atom.GetSymbol() for atom in mol.GetAtoms()}
            mask.append(not any(el in atoms for el in rare_elements))
        filtered_df = df[mask].reset_index(drop=True)
        print(f"Removed {len(df) - len(filtered_df)} molecule(s) due to rare elements.")
        return filtered_df, atom_summary

    # --- Filtering loop ---
    stats = {"invalid": [], "inorganic": [], "organometallic": [], "ion": [], "kept": []}

    for _, row in df.iterrows():
        smi = row[smiles_col]
        cano = canonicalize_smiles(smi)
        if cano is None:
            stats["invalid"].append(row)
            continue
        mol = Chem.MolFromSmiles(cano)
        if mol is None:
            stats["invalid"].append(row)
            continue
        if not is_organic(mol):
            row[smiles_col] = cano
            stats["inorganic"].append(row)
            continue
        if is_organometallic(mol):
            row[smiles_col] = cano
            stats["organometallic"].append(row)
            continue
        if is_ion(mol):
            row[smiles_col] = cano
            stats["ion"].append(row)
            continue
        row[smiles_col] = cano
        stats["kept"].append(row)

    # Build cleaned DataFrame
    cleaned_df = pd.DataFrame(stats["kept"]).drop_duplicates(subset=[smiles_col], keep="first")

    # Rare element filtering
    cleaned_df, atom_summary = remove_rare_elements(cleaned_df, smiles_col, threshold)

    # # Save
    # base, ext = os.path.splitext(output_path)
    # output_file = f"{base}_clean.csv"
    # cleaned_df.to_csv(output_file, index=False)
    # print(f"✅ Cleaned dataset saved to {output_file}")

    import inspect

    # --- Saving block ---
    # Try to get the dataframe variable name from the caller
    frame = inspect.currentframe().f_back
    df_name = None
    for var_name, var_val in frame.f_locals.items():
        if var_val is df:
            df_name = var_name
            break

    if df_name is None:
        df_name = "dataset"  # fallback if name not found

    # Ensure output_path is a directory
    if not os.path.isdir(output_path):
        os.makedirs(output_path, exist_ok=True)

    output_file = os.path.join(output_path, f"{df_name}_clean.csv")
    cleaned_df.to_csv(output_file, index=False)
    print(f"✅ Cleaned dataset saved to {output_file}")


    # Report
    print("Filtering summary:")
    for k in ["invalid", "inorganic", "organometallic", "ion", "kept"]:
        print(f"{k}: {len(stats[k])}")
    print(f"Original size: {len(df)}")
    print(f"After cleaning & rare element filter: {len(cleaned_df)}")

    return cleaned_df, atom_summary


Now we can apply that wrapped function above to a singular data frame with a column containing SMILES strings labeled as 'SMILES' or 'smiles'.

In [ ]:
SMILES_subset_c12_clean, atom_summary = clean_smiles_dataset(SMILES_subset, output_path=output_dir)

## **Section 5.0 Data Distributions and Curating a Dataset**

### ***Section 5.1** Looking at Trends in the Data*

#### **Section 5.11** Defining a function to identify functional groups present in the molecule in the dataset

In [ ]:
from rdkit import Chem
import pandas as pd

def analyze_molecules(df, smiles_column="SMILES"):
    """
    Adds functional group presence (True/False) to the dataframe.
    
    Returns:
        Updated DataFrame with new columns:
        - one column per functional group (True/False)
    """
    # Define common functional groups via SMARTS patterns
    functional_groups = {
        "Alcohol": "[CX4][OX2H]",                # primary/secondary alcohol
        "Amine": "[NX3;H2,H1;!$(NC=O)]",          # primary/secondary amine
        "CarboxylicAcid": "C(=O)[OH]",            # -COOH
        "Ester": "C(=O)OC",                       # ester
        "Amide": "C(=O)N",                        # amide
        "Ketone": "C(=O)C",                       # ketone
        "Aldehyde": "[CX3H1](=O)[#6]",            # aldehyde
        "Ether": "C-O-C",                         # ether
        "AromaticRing": "a1aaaaa1",               # 6-membered aromatic
        "Halogen": "[F,Cl,Br,I]"                  # any halogen
    }
    
    # Compile SMARTS patterns
    fg_smarts = {name: Chem.MolFromSmarts(smarts) for name, smarts in functional_groups.items()}
    
    fg_flags = {name: [] for name in functional_groups}
    
    for smi in df[smiles_column]:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            for name in functional_groups:
                fg_flags[name].append(None)
            continue
        
        for name, pattern in fg_smarts.items():
            fg_flags[name].append(mol.HasSubstructMatch(pattern))
    
    # Add results to DataFrame
    df = df.copy()
    for name in functional_groups:
        df[f"has_{name}"] = fg_flags[name]
    
    return df


analyzed_molecules = analyze_molecules(SMILES_subset_c12_clean)

#### **Section 5.12** Number of Atoms Distributions

After looking at the distribution of the number of atoms per molecule in this data set, I figured I'd truncate it to less than 40 atoms to help simplify the prediction.

In [ ]:
import matplotlib.pyplot as plt

# Create subplots with 1 row, 2 columns
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# First histogram (all data)
SMILES_subset_c12_clean["num_atoms"].hist(bins=150, ax=axes[0])
axes[0].set_title("All Molecules")
axes[0].set_xlabel("Number of Atoms")
axes[0].set_ylabel("Frequency")

# Apply filter for the second histogram
filtered_df = SMILES_subset_c12_clean[SMILES_subset_c12_clean["num_atoms"] < 150]

# Second histogram (filtered data)
filtered_df["num_atoms"].hist(bins=75, ax=axes[1])
axes[1].set_title("Molecules with < 150 atom including hydrogens")
axes[1].set_xlabel("Number of Atoms")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()


#### **Section 5.13** Looking at Distributions of TMIN and TMAX

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ranges = SMILES_subset_c12_clean[['TMIN', 'TMAX']].agg(['min', 'max'])
print(ranges)

# Sort the dataframe by TMIN for better visualization
df_sorted = SMILES_subset_c12_clean.sort_values(by="TMIN").reset_index(drop=True)

plt.figure(figsize=(10, 8))

# Plot each molecule's range as a horizontal line
for idx, row in df_sorted.iterrows():
    plt.hlines(y=idx, xmin=row['TMIN'], xmax=row['TMAX'], color='skyblue', alpha=0.7)
    plt.plot(row['TMIN'], idx, 'o', color='green', alpha=0.6)  # Start point
    plt.plot(row['TMAX'], idx, 'o', color='red', alpha=0.6)    # End point

plt.xlabel("Temperature (Celsius)")
plt.ylabel("Molecule index (sorted by TMIN)")
plt.title("Valid Temperature Ranges for Antoine Coefficients")
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(SMILES_subset_c12_clean['TMIN'], bins=30, color='green', alpha=0.7)
axes[0].set_title("TMIN Distribution")
axes[0].set_xlabel("TMIN (C)")
axes[0].set_ylabel("Count")

axes[1].hist(SMILES_subset_c12_clean['TMAX'], bins=30, color='red', alpha=0.7)
axes[1].set_title("TMAX Distribution")
axes[1].set_xlabel("TMAX (C)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()


## **Section 6.0 Further curation of the dataset**

### ***Section 6.1** Limiting the scope of the prediction.*

I am limiting the temperature range of predictions to between 0 celsius and 200 celsius. For tha reason, I am filtering the data based on if it's range of TMIN-TMAX (validity range of the Antoine coefficients). In the function below, I am dictating that at least 75% of a molecule's TMIN-TMAX range must be within 0 celsius and 200 celsius.

In [ ]:
import pandas as pd

# Filter based on overlap with 0-200°C
def overlap_fraction(row, lower=0, upper=200):
    # Original range length
    total_range = row["TMAX"] - row["TMIN"]
    if total_range <= 0:
        return 0
    # Overlap with 0-200
    overlap_start = max(row["TMIN"], lower)
    overlap_end = min(row["TMAX"], upper)
    overlap = max(0, overlap_end - overlap_start)
    return overlap / total_range

SMILES_subset_c12_clean["overlap_frac"] = SMILES_subset_c12_clean.apply(overlap_fraction, axis=1)
SMILES_subset_c12_clean_filtered = SMILES_subset_c12_clean[SMILES_subset_c12_clean["overlap_frac"] >= 0]

# Drop the overlap column we made, so things stay tidy.
SMILES_subset_c12_clean_filtered = SMILES_subset_c12_clean_filtered.drop(columns="overlap_frac")
SMILES_subset_c12_clean_filtered = SMILES_subset_c12_clean_filtered[~SMILES_subset_c12_clean_filtered["code"].astype(str).str.contains("3")]


### ***Section 6.2** Curating the Dataset for the Prediction of Vapor Pressure*

#### **Section 6.21** This subsection contains the functions necessary for determining what temperatures at which we will calculate vapor pressures. Call the correct one in the **Splitting the Data** subsection.

##### ***Section 6.211** Function for calculating vapor pressure at random temperatures sampled between TMIN and TMAX*

In [ ]:
import pandas as pd
import numpy as np

def generate_vapor_pressures_randomT(df, num_points=5, seed=None):
    """
    Expands Antoine equation data for each molecule over a temperature range
    using randomly selected points from TMIN to TMAX (inclusive).

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing at least: A, B, C, TMIN, TMAX
        Any other columns (chemical identifiers) will be preserved.
    num_points : int, optional
        Number of temperature points per molecule (default = 10).
    seed : int, optional
        Random seed for reproducibility.

    Returns
    -------
    pd.DataFrame
        Expanded DataFrame with original columns + T_C and P_mmHg.
        Number of rows = num_points * len(df)
    """
    if seed is not None:
        np.random.seed(seed)

    expanded_rows = []

    for _, row in df.iterrows():
        # Randomly select num_points temperatures between TMIN and TMAX
        T_range = np.random.uniform(low=row['TMIN'], high=row['TMAX'], size=num_points)

        for T_C in T_range:
            row_dict = row.to_dict()
            row_dict['T_C'] = T_C
            row_dict['P_mmHg'] = (row['A'] - row['B'] / (T_C + row['C']))
            expanded_rows.append(row_dict)

    return pd.DataFrame(expanded_rows)


##### ***Section 6.212** Function for calculating vapor pressure at random temperatures sampled between [TMIN, TMAX], [$0^{\circ}\text{C}$, TMAX], or [TMIN, $200^{\circ}\text{C}$].*

In [ ]:
import pandas as pd
import numpy as np

def generate_vapor_pressures_with_clamp_filtered_randomT(analyzed_molecules, num_points=4, Tmin_allowed=0, Tmax_allowed=200, max_pressure=5000, random_state=None):
    """
    Expands Antoine equation data for each molecule over a truncated valid temperature range.
    The range is based on the molecule's TMIN and TMAX, but clamped to the allowed Tmin/Tmax.
    Removes any molecules with P_mmHg exceeding max_pressure at any temperature.
    Temperatures are randomly selected instead of evenly spaced.
    
    Parameters
    ----------
    analyzed_molecules : pd.DataFrame
        Must contain: A, B, C, TMIN, TMAX.
        Other columns (e.g., SMILES) are preserved.
    num_points : int, optional
        Number of random temperature points per molecule (default = 4).
    Tmin_allowed : float, optional
        Minimum allowed temperature for calculations (default = 0°C).
    Tmax_allowed : float, optional
        Maximum allowed temperature for calculations (default = 200°C).
    max_pressure : float, optional
        Maximum allowable vapor pressure; molecules exceeding this are removed (default=5000 mmHg).
    random_state : int, optional
        Random seed for reproducibility.
    
    Returns
    -------
    pd.DataFrame
        Expanded and filtered DataFrame with original columns + T_C and P_mmHg.
    """
    if random_state is not None:
        np.random.seed(random_state)

    expanded_rows = []

    for _, row in analyzed_molecules.iterrows():
        # Clamp range
        Tmin = max(row['TMIN'], Tmin_allowed)
        Tmax = min(row['TMAX'], Tmax_allowed)

        if Tmin < Tmax:
            # Random temperatures in range
            T_range = np.random.uniform(Tmin, Tmax, size=num_points)
            for T_C in T_range:
                row_dict = row.to_dict()
                row_dict['T_C'] = T_C
                # Convert log10(P) to P_mmHg
                row_dict['P_mmHg'] = (row['A'] - row['B'] / (T_C + row['C']))
                expanded_rows.append(row_dict)

    expanded_df = pd.DataFrame(expanded_rows)

    # Filter out molecules where any P_mmHg exceeds max_pressure
    high_pressure_smiles = expanded_df.loc[expanded_df['P_mmHg'] > max_pressure, 'SMILES'].unique()
    filtered_df = expanded_df[~expanded_df['SMILES'].isin(high_pressure_smiles)].reset_index(drop=True)

    return filtered_df


#### ***Section 6.213** Function to filter the final dataset based on calculated pressures. The data will only contain molecules with vapor pressure less than 2000??? at $200^{\circ}\text{C}$.*

In [ ]:
import pandas as pd

def filter_by_p0(expanded_df, identifier_col="SMILES", p_thresh=760):
    """
    Filters an expanded vapor pressure DataFrame to keep only molecules
    where P_mmHg at 0°C is below a specified threshold.

    Parameters
    ----------
    expanded_df : pd.DataFrame
        DataFrame containing at least: identifier_col, T_C, P_mmHg
    identifier_col : str, optional
        Column used to identify unique molecules (default: "SMILES")
    p_thresh : float, optional
        Threshold for P_mmHg at 0°C (default: 760)

    Returns
    -------
    pd.DataFrame
        Filtered expanded DataFrame
    """
    # Get P_mmHg at 0°C for each molecule
    p0_df = expanded_df[expanded_df["T_C"] == 0][[identifier_col, "P_mmHg"]]

    # Identify molecules below the threshold
    valid_molecules = p0_df[p0_df["P_mmHg"] < p_thresh][identifier_col].unique()

    # Filter the full expanded_df
    filtered_df = expanded_df[expanded_df[identifier_col].isin(valid_molecules)].copy()

    print(f"Number of rows after filtering: {len(filtered_df)}")
    return filtered_df

# Example usage:
# filtered_expanded_df = filter_by_p0(expanded_df)
# filtered_expanded_df.head(10)


## **Section 7.0 Splitting and preparing the data in the format accepted by Chemprop**

Chemprop has some very specific requirements for how the data needs to be fed (or how the columns should be referenced in the call) that necessitate a little leg work on our end. No such thing as a free lunch I suppose. It's not too difficult, though. We just need to create a .csv file that contains a SMILES string and corresponding target of our prediction (Pressure) with the SMILES string being in the first column. Additionally, we need to convert the temperatures into a .npz file because that is what Chemprop requests.

Due to the nature of this prediction requiring the data expansion performed above, we need to also include a column which contains the corresponding split that the data point belongs to. All of this preparation was done in the above cell, but I'm making not of it again to highlight its importance. In the below cell, we are simply concatenating the individual train, validation, and test splits.

In this section, it is necessary to note that we are splitting the data before applying the vapor pressure calculation function shown above. We do this because there would be data leakage if we duplicated each molecule *num_points* times and THEN split the data. We could have one molecule (as 10 separate data points) spread out across the train, validation, and/or test sets. That would make our model look great, but it would give more confidence in the prediction than it warrents for unseen compounds. 

For that reason, we split the molecules into their respective sets, THEN we applying the vapor pressure calculation function to avoid data leakage.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os

# Split the data before the great expansion
train_df, temp_df = train_test_split(SMILES_subset_c12_clean_filtered, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Expand the splits by adding vapor pressure data
train_expanded = generate_vapor_pressures_randomT(train_df, num_points=5)
val_expanded = generate_vapor_pressures_randomT(val_df, num_points=5)
test_expanded = generate_vapor_pressures_randomT(test_df, num_points=5)

# Add 'split' column
train_expanded["split"] = "train"
val_expanded["split"] = "val"
test_expanded["split"] = "test"

# Check sizes
print("Sanity Check: Each split should have increased by num_points times the original size\n")
print(f"Train: {train_df.shape[0]} → {train_expanded.shape[0]}")
print(f"Val: {val_df.shape[0]} → {val_expanded.shape[0]}")
print(f"Test: {test_df.shape[0]} → {test_expanded.shape[0]}")

# Folder to save files
folder = "../Datasets/trainable_data"
os.makedirs(folder, exist_ok=True)  # create folder if it doesn't exist

master_df = pd.concat([train_expanded[['SMILES','P_mmHg','split']], val_expanded[['SMILES','P_mmHg','split']], test_expanded[['SMILES','P_mmHg','split']]], ignore_index=True)
master_df.to_csv(os.path.join(folder, "VP_chemprop_train_file.csv"), index=False)

master_feat = pd.concat([train_expanded[['SMILES', 'T_C','split']], val_expanded[['SMILES', 'T_C','split']], test_expanded[['SMILES', 'T_C','split']]], ignore_index=True)
master_feat.to_csv(os.path.join(folder, "VP_chemprop_feat_file.csv"), index=False) # This won't be the file called in Chemprop (that's the .npz below)

# Chemprop requests that the extra features be fed as a .npz file, so we have to convert it.
# Extract T_C column as a NumPy array (2D shape: n_samples x 1)
T_C_array = master_feat["T_C"].to_numpy().reshape(-1, 1)

# Save as .npz for Chemprop
np.savez(os.path.join(folder,"VP_chemprop_feat_file.npz"), T_C_array)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

# Split the data before the great expansion
train_df, temp_df = train_test_split(!!!DATAFRAME OF CLEANED DATA!!!, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Add 'split' column
train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

output_dir = "../Datasets/trainable_data"
os.makedirs(output_dir, exist_ok=True)

train_val_test_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
train_val_test_df.to_csv(os.path.join(output_dir, !!!VP_chemprop_train_file.csv USE THIS AS NAMING FORMAT!!!), index=False)